### Imports and Configuration

In [ ]:
import cv2
import mediapipe as mp
import os
import re
import json
import copy
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

# --- Configuration ---
INPUT_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")

# --- Initialize MediaPipe Pose ---
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose = mp_pose.Pose(
    static_image_mode=False, 
    model_complexity=2, 
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

I0000 00:00:1777673349.306007 8912226 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2


W0000 00:00:1777673349.478565 10412783 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777673349.551662 10412785 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


### Core Extraction Function

In [3]:
def process_video(video_path, output_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return False

    video_data = {
        "video_path": str(video_path),
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "frames": []
    }

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        
        frame_data = {"frame_index": frame_idx, "landmarks": None}
        
        if results.pose_world_landmarks:
            landmarks = []
            for lm in results.pose_world_landmarks.landmark:
                landmarks.append({'x': lm.x, 'y': lm.y, 'z': lm.z, 'visibility': lm.visibility})
            frame_data["landmarks"] = landmarks
        
        video_data["frames"].append(frame_data)
        frame_idx += 1

    cap.release()

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w') as f:
        json.dump(video_data, f)
        
    return True

### Sanity Check Visualization

In [4]:
target_videos = []

# rglob('*') searches recursively, ignoring how deep the person# folders go
for video_path in INPUT_DIR.rglob('*'):
    if video_path.is_file() and video_path.suffix.lower() in ['.mp4', '.avi', '.mov', '.mkv']:
        # Extract the exercise class
        try:
            raw_idx = video_path.parts.index('raw')
            exercise_class = video_path.parts[raw_idx + 1]
            
            target_videos.append({
                "path": video_path,
                "exercise": exercise_class
            })
        except ValueError:
            continue

print(f"Found {len(target_videos)} videos in the raw directory.")

Found 800 videos in the raw directory.


### Batch Processing Execution

In [5]:
processed_count, skipped_count, error_count = 0, 0, 0

for vid_info in tqdm(target_videos, desc="Extracting Landmarks"):
    video_path = vid_info["path"]
    exercise_class = vid_info["exercise"]
    
    # Create output directory
    class_output_dir = OUTPUT_DIR / exercise_class
    class_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Append the parent folder name to prevent naming collisions
    safe_filename = f"{video_path.parent.name}_{video_path.stem}.json"
    output_path = class_output_dir / safe_filename
    
    if not output_path.exists():
        success = process_video(video_path, output_path)
        if success:
            processed_count += 1
        else:
            error_count += 1
    else:
        skipped_count += 1

print(f"\nProcessed: {processed_count} | Skipped: {skipped_count} | Errors: {error_count}")

Extracting Landmarks:   0%|          | 0/800 [00:00<?, ?it/s]

W0000 00:00:1777641609.450021 8917292 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777641609.500503 8917299 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



Processed: 700 | Skipped: 100 | Errors: 0


### Dataset Balance Checker

In [51]:
PROCESSED_DIR = Path("../data/processed")
TARGET_COUNT = 250

def check_balance_and_augmentation(data_dir, target_count):
    if not data_dir.exists():
        print(f"Error: Directory '{data_dir}' not found. Please verify the path.")
        return

    counts = {}
    
    # Iterate through each exercise directory
    for exercise_folder in data_dir.iterdir():
        # Exclude hidden system files or directories
        if exercise_folder.is_dir() and not exercise_folder.name.startswith('.'):
            exercise_name = exercise_folder.name
            
            # Recursively count all JSON files within the exercise directory
            json_count = len(list(exercise_folder.rglob("*.json")))
            counts[exercise_name] = json_count

    # Format and display the results
    if counts:
        df = pd.DataFrame(list(counts.items()), columns=['Exercise Class', 'Valid JSONs'])
        df = df.sort_values(by='Valid JSONs', ascending=False).reset_index(drop=True)
        
        print("=========================================")
        print(f"       DATASET BALANCE REPORT            ")
        print(f"       (Target: {target_count} per class)")
        print("=========================================")
        print(df.to_string())
        print("-----------------------------------------\n")
        
        total_videos = df['Valid JSONs'].sum()
        print(f"Total processed files: {total_videos}\n")
        
        print("=========================================")
        print("         AUGMENTATION TARGETS            ")
        print("=========================================")
        for index, row in df.iterrows():
            current_count = row['Valid JSONs']
            shortfall = target_count - current_count
            exercise_name = row['Exercise Class']
            
            if shortfall <= 0:
                print(f"Target Met: '{exercise_name}' contains {current_count} instances. No augmentation required.")
            else:
                if current_count > 0:
                    multiplier = target_count / current_count
                    print(f"Deficit: '{exercise_name}' requires {shortfall} additional instances. (Expansion multiplier: {multiplier:.2f}x)")
                else:
                    print(f"Deficit: '{exercise_name}' contains 0 instances. Cannot calculate expansion multiplier.")
    else:
        print("Notice: No JSON files were found in the specified directory.")

# Execute the function
check_balance_and_augmentation(PROCESSED_DIR, TARGET_COUNT)

       DATASET BALANCE REPORT            
       (Target: 250 per class)
  Exercise Class  Valid JSONs
0        pull_up          250
1          squat          250
2        push_up          250
3    bench_press          250
-----------------------------------------

Total processed files: 1000

         AUGMENTATION TARGETS            
Target Met: 'pull_up' contains 250 instances. No augmentation required.
Target Met: 'squat' contains 250 instances. No augmentation required.
Target Met: 'push_up' contains 250 instances. No augmentation required.
Target Met: 'bench_press' contains 250 instances. No augmentation required.


### Automated Data Augmentation Pipeline

In [ ]:
PROCESSED_DIR = Path("../data/processed")
TARGET_COUNT = 250

def augment_noise(data):
    """Injects minor Gaussian noise to simulate tracking inaccuracies."""
    aug_data = copy.deepcopy(data)
    for frame in aug_data.get('frames', []):
        if not isinstance(frame, dict):
            continue
            
        landmarks = frame.get('landmarks')
        # Skip if landmarks is None or empty
        if not landmarks:
            continue
            
        for lm in landmarks:
            if isinstance(lm, dict) and all(k in lm for k in ('x', 'y', 'z')):
                lm['x'] += random.gauss(0, 0.01)
                lm['y'] += random.gauss(0, 0.01)
                lm['z'] += random.gauss(0, 0.01)
    return aug_data

def augment_speed(data):
    """Drops alternating frames to simulate a faster execution speed."""
    aug_data = copy.deepcopy(data)
    frames = aug_data.get('frames', [])
    if isinstance(frames, list):
        aug_data['frames'] = frames[::2]
    return aug_data

def augment_flip(data):
    """Mirrors the X coordinates and swaps left/right MediaPipe topology."""
    aug_data = copy.deepcopy(data)
    
    swap_pairs = [
        (1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12), (13, 14), 
        (15, 16), (17, 18), (19, 20), (21, 22), (23, 24), (25, 26), 
        (27, 28), (29, 30), (31, 32)
    ]
    
    for frame in aug_data.get('frames', []):
        if not isinstance(frame, dict):
            continue
            
        landmarks = frame.get('landmarks')
        if not landmarks: 
            continue
            
        for lm in landmarks:
            if isinstance(lm, dict) and 'x' in lm:
                lm['x'] = 1.0 - lm['x']
            
        for left_idx, right_idx in swap_pairs:
            if left_idx < len(landmarks) and right_idx < len(landmarks):
                left_lm = landmarks[left_idx]
                right_lm = landmarks[right_idx]
                
                if isinstance(left_lm, dict) and isinstance(right_lm, dict):
                    temp_x, temp_y, temp_z = left_lm.get('x'), left_lm.get('y'), left_lm.get('z')
                    
                    left_lm['x'], left_lm['y'], left_lm['z'] = right_lm.get('x'), right_lm.get('y'), right_lm.get('z')
                    right_lm['x'], right_lm['y'], right_lm['z'] = temp_x, temp_y, temp_z
                
    return aug_data

def balance_dataset(data_dir, target_count):
    if not data_dir.exists():
        print(f"Error: Directory '{data_dir}' not found.")
        return

    augmentation_methods = [
        ('noise', augment_noise),
        ('speed', augment_speed),
        ('flip', augment_flip)
    ]

    print("=========================================")
    print("       DATASET BALANCING PROCESS         ")
    print("=========================================")

    for exercise_folder in data_dir.iterdir():
        if exercise_folder.is_dir() and not exercise_folder.name.startswith('.'):
            
            all_jsons = list(exercise_folder.rglob("*.json"))
            current_count = len(all_jsons)
            
            # --- Downsampling Surplus ---
            if current_count > target_count:
                surplus = current_count - target_count
                print(f"Downsampling '{exercise_folder.name}': Removing {surplus} surplus files...")
                
                # Exclude previously augmented files from deletion if possible, prioritize removing originals
                # or randomly sample from the entire pool
                files_to_delete = random.sample(all_jsons, surplus)
                
                for file_path in files_to_delete:
                    file_path.unlink() # Deletes the file
                
                print(f"Completed '{exercise_folder.name}': Reduced to {target_count} files.")
                continue

            # --- Augmenting Deficits ---
            shortfall = target_count - current_count
            
            if shortfall == 0:
                print(f"Skipping '{exercise_folder.name}': Target already perfectly met ({current_count}).")
                continue
                
            # Isolate original files strictly by checking the filename string
            original_jsons = [f for f in all_jsons if "_aug_" not in f.name]
            
            if not original_jsons:
                print(f"Error: No pristine source files in '{exercise_folder.name}' to augment.")
                continue

            print(f"Augmenting '{exercise_folder.name}': Generating {shortfall} new files...")
            
            generated_count = 0
            while generated_count < shortfall:
                source_file = random.choice(original_jsons)
                method_name, method_func = random.choice(augmentation_methods)
                
                with open(source_file, 'r') as f:
                    try:
                        data = json.load(f)
                    except json.JSONDecodeError:
                        continue
                
                augmented_data = method_func(data)
                
                new_filename = f"{source_file.stem}_aug_{generated_count}_{method_name}.json"
                new_filepath = exercise_folder / new_filename
                
                with open(new_filepath, 'w') as f:
                    json.dump(augmented_data, f, indent=4)
                    
                generated_count += 1
                
            print(f"Completed '{exercise_folder.name}': Reached {target_count} files.")

# Execute the balancing pipeline
balance_dataset(PROCESSED_DIR, TARGET_COUNT)

       DATASET BALANCING PROCESS         
Augmenting 'pull_up': Generating 122 new files...
Completed 'pull_up': Reached 250 files.
Downsampling 'squat': Removing 1 surplus files...
Completed 'squat': Reduced to 250 files.
Downsampling 'push_up': Removing 20 surplus files...
Completed 'push_up': Reduced to 250 files.
Augmenting 'bench_press': Generating 99 new files...
Completed 'bench_press': Reached 250 files.


### Dataset Flattening

In [ ]:
OUTPUT_DIR = Path("../data/processed")
FINAL_DIR = Path("../data/final")

def get_dataset_source(original_base):
    """
    Categorizes the file format to namespace duplicate person tags.
    Prevents 'person1' in the STU dataset from merging with 'person1' in the standard dataset.
    """
    base_lower = original_base.lower()
    
    if 'stu' in base_lower:
        return 'stu_data'
    else:
        return 'standard_data'

def get_unique_person_key(original_base, exercise_name):
    """
    Globally identifies a person across ALL dataset formats.
    Namespaces explicit 'person#' tags by their source dataset.
    """
    original_base_lower = original_base.lower()

    # Explicit Person Match (Namespaced by Source)
    person_match = re.search(r'(person\s*\d+)', original_base_lower)
    if person_match:
        person_id = person_match.group(1).replace(" ", "")
        source_namespace = get_dataset_source(original_base_lower)
        return f"{person_id}_{source_namespace}"

    # Untagged Single Instances: If no 'person#' tag exists, it's a unique file
    clean_base = original_base_lower
    
    clean_base = re.sub(r'[_\-\s]+(cam|camera|view|trial|angle|c)\s*\d+', '', clean_base)
    clean_base = re.sub(r'[_\-\s]+(front|side|back|rgb|kinect|gopro)', '', clean_base)
    clean_base = clean_base.strip('_- ')

    return f"single_{clean_base}"


def create_flattened_dataset(source_dir, target_dir):
    if not source_dir.exists():
        print(f"error: source directory '{source_dir}' not found.")
        return

    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    for exercise_folder in source_dir.iterdir():
        if exercise_folder.is_dir() and not exercise_folder.name.startswith('.'):
            exercise_name = exercise_folder.name.replace(" ", "_")
            final_exercise_dir = target_dir / exercise_name
            final_exercise_dir.mkdir(parents=True, exist_ok=True)
            
            all_jsons = sorted(list(exercise_folder.rglob("*.json")))
            print(f"transferring '{exercise_name}' ({len(all_jsons)} files)...")
            
            base_file_to_person_key = {}
            person_key_counts = {}
            
            for json_file in all_jsons:
                original_stem = json_file.stem
                
                if "_aug_" in original_stem:
                    original_base = original_stem.split("_aug_")[0]
                else:
                    original_base = original_stem
                    
                if original_base not in base_file_to_person_key:
                    unique_key = get_unique_person_key(original_base, exercise_name)
                    base_file_to_person_key[original_base] = unique_key
                    person_key_counts[unique_key] = person_key_counts.get(unique_key, 0) + 1

            debug_log_path = final_exercise_dir / f"{exercise_name}_debug_log.txt"
            
            key_to_files_map = {}
            for base_file, key in base_file_to_person_key.items():
                if key not in key_to_files_map:
                    key_to_files_map[key] = []
                key_to_files_map[key].append(base_file)

            with open(debug_log_path, "w", encoding="utf-8") as debug_file:
                debug_file.write(f"=== DEBUG REPORT: {exercise_name} ===\n")
                debug_file.write(f"Total Base Files: {len(base_file_to_person_key)}\n")
                debug_file.write(f"Total Unique People Identified: {len(person_key_counts)}\n")
                debug_file.write("="*40 + "\n\n")
                
                for key in sorted(key_to_files_map.keys()):
                    files = key_to_files_map[key]
                    debug_file.write(f"GROUP KEY: [{key}] -> contains {len(files)} file(s)\n")
                    for f in sorted(files):
                        debug_file.write(f"   - {f}\n")
                    debug_file.write("\n")

            exercise_person_mapping = {}
            person_counter = 1
            for key, count in person_key_counts.items():
                if count > 1:
                    exercise_person_mapping[key] = f"person{person_counter}"
                    person_counter += 1

            base_name_mapping = {}
            file_counter = 1
            
            for json_file in tqdm(all_jsons, desc=exercise_name, unit="file"):
                original_stem = json_file.stem
                
                if "_aug_" in original_stem:
                    parts = original_stem.split("_aug_")
                    original_base = parts[0] 
                    aug_suffix = f"_aug_{parts[1]}"
                else:
                    original_base = original_stem
                    aug_suffix = ""
                
                if original_base not in base_name_mapping:
                    unique_person_key = base_file_to_person_key[original_base]
                    
                    if unique_person_key in exercise_person_mapping:
                        person_tag = f"_{exercise_person_mapping[unique_person_key]}"
                    else:
                        person_tag = "" 
                    
                    base_name_mapping[original_base] = f"{exercise_name}_{file_counter:03d}{person_tag}"
                    file_counter += 1
                    
                new_base_name = base_name_mapping[original_base]
                new_name = f"{new_base_name}{aug_suffix}.json"
                destination_path = final_exercise_dir / new_name
                    
                shutil.copy2(json_file, destination_path)

    print("\nclean dataset is now ready at:", target_dir.resolve())

# execute the process
create_flattened_dataset(OUTPUT_DIR, FINAL_DIR)

transferring 'pull_up' (250 files)...


pull_up:   0%|          | 0/250 [00:00<?, ?file/s]

transferring 'squat' (250 files)...


squat:   0%|          | 0/250 [00:00<?, ?file/s]

transferring 'push_up' (250 files)...


push_up:   0%|          | 0/250 [00:00<?, ?file/s]

transferring 'bench_press' (250 files)...


bench_press:   0%|          | 0/250 [00:00<?, ?file/s]


clean dataset is now ready at: /Users/renzo/Projects/CoreSet-AI-Fitness-Tracker-ML-Pipeline/data/final


In [ ]:
import json
import math
from pathlib import Path
from collections import Counter

INPUT_ROOT = Path("src/data/final")
OUTPUT_ROOT = Path("src/data/interpolated_json")
CATEGORIES = ["bench_press", "pull_up", "push_up", "squat"]


# -----------------------------
# 1. Validate file parity
# -----------------------------
print("FILE PARITY AUDIT")

total_input = 0
total_output = 0

for category in CATEGORIES:
    input_files = sorted((INPUT_ROOT / category).glob("*.json"))
    output_files = sorted((OUTPUT_ROOT / category).glob("*.json"))

    input_names = {p.name for p in input_files}
    output_names = {p.name for p in output_files}

    missing = input_names - output_names
    extra = output_names - input_names

    total_input += len(input_files)
    total_output += len(output_files)

    print(
        f"{category:<12} "
        f"input={len(input_files)} "
        f"output={len(output_files)} "
        f"missing={len(missing)} "
        f"extra={len(extra)}"
    )

    if missing:
        print("  Missing:", sorted(missing)[:10])
    if extra:
        print("  Extra:", sorted(extra)[:10])

print(f"\ntotal input={total_input}")
print(f"total output={total_output}")

In [ ]:
# -----------------------------
# 2. Validate interpolated JSON values
# -----------------------------
def validate_interpolated_json(root):
    summary = Counter()
    category_summary = {}
    examples = []

    for category_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        counts = Counter(files=0)

        for path in sorted(category_dir.glob("*.json")):
            counts["files"] += 1

            try:
                data = json.loads(path.read_text())
            except Exception as exc:
                counts["bad_json"] += 1
                examples.append((str(path), "bad_json", str(exc)))
                continue

            frames = data.get("frames", data) if isinstance(data, dict) else data

            if not isinstance(frames, list):
                counts["bad_frames"] += 1
                examples.append((str(path), "bad_frames", type(frames).__name__))
                continue

            for frame_idx, frame in enumerate(frames):
                landmarks = frame.get("landmarks") if isinstance(frame, dict) else None

                if landmarks is None:
                    counts["null_landmark_frames"] += 1
                    examples.append((str(path), frame_idx, "landmarks_null"))
                    continue

                if len(landmarks) != 33:
                    counts["wrong_landmark_count_frames"] += 1
                    examples.append((str(path), frame_idx, f"landmarks={len(landmarks)}"))

                for lm_idx, landmark in enumerate(landmarks):
                    if landmark is None:
                        counts["null_landmarks"] += 1
                        examples.append((str(path), frame_idx, lm_idx, "landmark_null"))
                        continue

                    if isinstance(landmark, dict):
                        values = [
                            landmark.get("x"),
                            landmark.get("y"),
                            landmark.get("z"),
                            landmark.get("visibility"),
                        ]
                    else:
                        values = list(landmark[:4])

                    try:
                        vals = [float(v) for v in values]
                    except Exception:
                        counts["non_numeric_values"] += 1
                        examples.append((str(path), frame_idx, lm_idx, "non_numeric", values))
                        continue

                    non_finite = sum(not math.isfinite(v) for v in vals)
                    counts["non_finite_values"] += non_finite

                    if non_finite:
                        examples.append((str(path), frame_idx, lm_idx, "non_finite", vals))

                    if all(math.isfinite(v) and v == 0.0 for v in vals[:3]):
                        counts["all_zero_xyz_triplets"] += 1
                        examples.append((str(path), frame_idx, lm_idx, "zero_xyz", vals))

        category_summary[category_dir.name] = counts
        summary.update(counts)

    return category_summary, summary, examples

In [ ]:
category_summary, summary, examples = validate_interpolated_json(OUTPUT_ROOT)

print("\nCATEGORY VALUE AUDIT")

for category, counts in category_summary.items():
    print(
        f"{category}: "
        f"files={counts['files']}, "
        f"bad_json={counts['bad_json']}, "
        f"null_frames={counts['null_landmark_frames']}, "
        f"null_landmarks={counts['null_landmarks']}, "
        f"zero_xyz={counts['all_zero_xyz_triplets']}, "
        f"non_finite={counts['non_finite_values']}, "
        f"wrong_lm_frames={counts['wrong_landmark_count_frames']}"
    )

print("\nTOTALS")
for key in [
    "files",
    "bad_json",
    "bad_frames",
    "null_landmark_frames",
    "null_landmarks",
    "all_zero_xyz_triplets",
    "non_finite_values",
    "non_numeric_values",
    "wrong_landmark_count_frames",
]:
    print(f"{key}: {summary[key]}")

print("\nEXAMPLES")
if examples:
    for example in examples[:20]:
        print(example)
else:
    print("none")